# 演習 03 — PyTorch で FashionMNIST 風ネットワークを学習する

**目標**: CNN を実装して FashionMNIST を分類し、テスト精度 ≥ 85% を目指す。  
依存: `pip install torch torchvision matplotlib`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}, デバイス: {device}")

CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

## 1. データの準備

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_ds = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform_train)
test_ds  = datasets.FashionMNIST("./data", train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2)

print(f"訓練: {len(train_ds):,} サンプル, テスト: {len(test_ds):,} サンプル")

In [ ]:
# サンプルを可視化
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for ax, img, label in zip(axes.flat, images[:16], labels[:16]):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(CLASSES[label], fontsize=8)
    ax.axis("off")
plt.suptitle("FashionMNIST サンプル", fontsize=12)
plt.tight_layout()
plt.show()

## 2. CNN モデルの定義

In [ ]:
class FashionCNN(nn.Module):
    """FashionMNIST 分類用 CNN。"""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        # 特徴抽出部
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28×28 → 28×28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28×28 → 14×14
            nn.Dropout2d(0.1),

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 14×14 → 14×14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14×14 → 7×7
            nn.Dropout2d(0.2),
        )
        # 分類部
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


model = FashionCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"総パラメータ数: {total_params:,}")

# フォワードパスの確認
dummy = torch.randn(4, 1, 28, 28).to(device)
print(f"出力形状: {model(dummy).shape}")

## 3. 訓練

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-2, steps_per_epoch=len(train_loader), epochs=10
)

train_losses, test_accs = [], []
N_EPOCHS = 10

for epoch in range(N_EPOCHS):
    # 訓練
    model.train()
    epoch_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item() * len(imgs)
    train_losses.append(epoch_loss / len(train_ds))

    # テスト評価
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
    acc = correct / len(test_ds)
    test_accs.append(acc)
    print(f"Epoch {epoch+1:2d}/{N_EPOCHS}: 損失={train_losses[-1]:.4f}, "
          f"テスト精度={acc:.4f} {'✅' if acc >= 0.85 else ''}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, color="steelblue")
axes[0].set_title("訓練損失")
axes[0].set_xlabel("エポック")

axes[1].plot(test_accs, color="coral")
axes[1].axhline(0.85, color="gray", linestyle="--", label="目標 85%")
axes[1].set_title("テスト精度")
axes[1].set_xlabel("エポック")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"最終テスト精度: {test_accs[-1]:.4f}")

## 4. 誤分類の分析

In [ ]:
model.eval()
wrong_imgs, wrong_preds, wrong_labels = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = model(imgs).argmax(1)
        mask = preds != labels
        wrong_imgs.extend(imgs[mask].cpu())
        wrong_preds.extend(preds[mask].cpu())
        wrong_labels.extend(labels[mask].cpu())
        if len(wrong_imgs) >= 10:
            break

print(f"誤分類サンプル数（先頭 10 件）:")
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, img, pred, true in zip(axes.flat, wrong_imgs[:10],
                                wrong_preds[:10], wrong_labels[:10]):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"予測: {CLASSES[pred]}\n正解: {CLASSES[true]}",
                 fontsize=8, color="red")
    ax.axis("off")
plt.suptitle("誤分類サンプル", fontsize=12)
plt.tight_layout()
plt.show()